# ML Audit Agent Evaluation

Evaluates the LangFlow ML Audit Agent using MLflow's GenAI evaluation with LLM-as-judge scoring.

**Approach:** Simple questions that test agent capabilities without heavy data queries.

In [1]:
!pip install -q 'mlflow>=3.3' requests pandas anthropic



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import json
import requests
import mlflow
from mlflow.genai import scorer

LANGFLOW_URL = os.environ.get("LANGFLOW_URL", "http://langflow:7860")
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000")
MLFLOW_UI_URL = "http://127.0.0.1:5001"  # External URL for browser access

print(f"LangFlow: {LANGFLOW_URL}")
print(f"MLflow API: {MLFLOW_TRACKING_URI}")
print(f"MLflow UI: {MLFLOW_UI_URL}")

LangFlow: http://langflow:7860
MLflow API: http://mlflow:5000
MLflow UI: http://127.0.0.1:5001


## 1. LangFlow Agent Client

In [3]:
import time

class LangFlowAgent:
    """Simple LangFlow agent client with retry logic."""
    
    def __init__(self, base_url: str = None):
        self.base_url = base_url or LANGFLOW_URL
        self.flow_id = None
        self.token = None
        self._init()
    
    def _init(self):
        try:
            r = requests.get(f"{self.base_url}/api/v1/auto_login", timeout=10)
            if r.ok:
                self.token = r.json().get("access_token")
            
            headers = {"Authorization": f"Bearer {self.token}"} if self.token else {}
            r = requests.get(f"{self.base_url}/api/v1/flows/", headers=headers, timeout=10)
            if r.ok:
                for flow in r.json():
                    if "audit" in flow.get("name", "").lower():
                        self.flow_id = flow["id"]
                        print(f"Found flow: {flow['name']}")
                        break
        except Exception as e:
            print(f"Init error: {e}")
    
    def ask(self, question: str, timeout: int = 180, retries: int = 3) -> str:
        """Ask the agent a question with retry logic."""
        if not self.flow_id:
            return "Error: No audit flow found"
        
        headers = {"Content-Type": "application/json"}
        if self.token:
            headers["Authorization"] = f"Bearer {self.token}"
        
        last_error = None
        for attempt in range(retries):
            try:
                r = requests.post(
                    f"{self.base_url}/api/v1/run/{self.flow_id}",
                    json={"input_value": question, "output_type": "chat", "input_type": "chat"},
                    headers=headers,
                    timeout=timeout
                )
                
                if not r.ok:
                    last_error = f"HTTP {r.status_code}"
                    continue
                
                result = r.json()
                try:
                    return result["outputs"][0]["outputs"][0]["results"]["message"]["text"]
                except (KeyError, IndexError):
                    return json.dumps(result, indent=2)
                    
            except requests.exceptions.Timeout:
                last_error = "Request timed out"
            except Exception as e:
                last_error = str(e)
            
            if attempt < retries - 1:
                print(f"  Retry {attempt + 1}/{retries - 1} after error: {last_error}")
                time.sleep(2)
        
        return f"Error: {last_error}"

agent = LangFlowAgent()
print(f"Flow ID: {agent.flow_id}")

# Test connectivity
print("\nTesting LangFlow connection...")
test_response = agent.ask("Hello", timeout=30, retries=1)
if test_response.startswith("Error:"):
    print(f"WARNING: LangFlow not responding - {test_response}")
else:
    print(f"LangFlow OK - got {len(test_response)} chars")

Found flow: ML Algorithm Audit Agent
Flow ID: 4f8f5f50-aedf-4edc-88ae-7867d957668d

Testing LangFlow connection...


## 2. Evaluation Dataset

Simple questions that test agent understanding without triggering large queries.

In [4]:
EVAL_DATASET = [
    {
        "inputs": {"question": "What metrics do you use to evaluate ML fraud detection performance?"},
        "expectations": {"expected_response": "Agent should explain precision, recall, F1, and their relevance to fraud detection."},
    },
    {
        "inputs": {"question": "What is the difference between a false positive and false negative in fraud detection?"},
        "expectations": {"expected_response": "Agent should explain both concepts and their business impact."},
    },
    {
        "inputs": {"question": "How would you investigate if the ML model is missing fraudulent transactions?"},
        "expectations": {"expected_response": "Agent should describe an investigation approach - checking recall, analyzing false negatives."},
    },
    {
        "inputs": {"question": "What could cause ML and rule-based detection to disagree?"},
        "expectations": {"expected_response": "Agent should discuss model drift, threshold differences, or rule coverage gaps."},
    },
    {
        "inputs": {"question": "Summarize the current fraud detection status in one sentence."},
        "expectations": {"expected_response": "Agent should provide a brief, actionable summary."},
    },
]

print(f"Dataset: {len(EVAL_DATASET)} questions")

Dataset: 5 questions


## 3. LLM-as-Judge Scorer

Uses Claude/Anthropic to evaluate response quality.

In [5]:
import os
import anthropic
import json
import re
ANTHROPIC_AVAILABLE = False

def get_claude_response(prompt: str, max_tokens: int = 200):
    try:
        # Initialize the client. It automatically looks for the ANTHROPIC_API_KEY env variable.
        # Alternatively, pass it directly: client = anthropic.Anthropic(api_key="your-api-key")
        api_key = os.getenv("ANTHROPIC_API_KEY", "")
        client = anthropic.Anthropic(api_key=api_key)
        
        # Call the direct Messages API
        # Note: Direct Anthropic API uses direct model strings (e.g., 'claude-3-5-sonnet-latest')
        response = client.messages.create(
            model="claude-sonnet-4-6", 
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        
        # The SDK parses the object automatically
        text = "".join([block.text for block in response.content if block.type == "text"])
        match = re.search(r'\{[^}]+\}', text)
        if match:
            return json.loads(match.group())
        return {"overall": 3, "reason": "Could not parse judge response"}
                
    except anthropic.APIConnectionError as e:
        return {"overall": 0, "reason": f"Server could not be reached: {e}"}
    except anthropic.RateLimitError as e:
        return {"overall": 0, "reason": f"Rate limit hit: {e}"}
    except anthropic.APIStatusError as e:
        return {"overall": 0, "reason": f"API error (Status {e.status_code}): {e.message}"}
    except Exception as e:
        return {"overall": 0, "reason": f"An unexpected error occurred: {e}"}

def llm_judge(question: str, response: str, expectations: str) -> dict:
    """Use Claude to judge response quality."""
    prompt = f"""You are evaluating an ML Audit Agent's response quality.

Question asked: {question}
Expected context: {expectations}
Agent's response:
{response}

Rate the response on these criteria (1-5 scale):
1. Relevance: Does it address the question?
2. Accuracy: Is the information correct?
3. Helpfulness: Would this help a fraud analyst?

Respond in JSON format:
{{"relevance": <1-5>, "accuracy": <1-5>, "helpfulness": <1-5>, "overall": <1-5>, "reason": "<brief explanation>"}}"""
    
    result = get_claude_response(prompt, max_tokens=400)
    # If get_claude_response encountered an API error, it returned a dict. 
    # We can just return that error dict directly.
    if isinstance(result, dict):
        return result
        
    # Otherwise, result is a string, so we parse out the JSON object
    try:
        match = re.search(r'\{[^}]+\}', result)
        if match:
            return json.loads(match.group())
        return {"overall": 3, "reason": f"Could not parse judge response. Raw text: {result[:100]}"}
        
    except Exception as e:
        return {"overall": 0, "reason": f"Judge parsing error: {e}"}

# Test Anthropic availability
print("Testing Anthropic credentials...")
try:
    test_result = llm_judge("test", "test response", "test expectation")
    if test_result.get("overall", 0) > 0:
        ANTHROPIC_AVAILABLE = True

        print(f"Anthropic available - LLM judge enabled")
    else:
        print(f"Anthropic test failed: {test_result.get('reason')}")
except Exception as e:
    print(f"Anthropic test error: {e}")


Testing Anthropic credentials...
Anthropic test failed: An unexpected error occurred: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"


In [6]:
@scorer
def response_quality(inputs: dict, outputs: str, expectations: dict) -> float:
    """LLM-as-judge scorer for response quality (0-1 scale)."""
    if not ANTHROPIC_AVAILABLE:
        raise ConnectionError  # Skip this scorer if ANTHROPIC not available
    if not outputs or outputs.startswith("Error:"):
        return 0.0
    
    expected = expectations.get("expected_response", "")
    judgment = llm_judge(inputs.get("question", ""), outputs, expected)
    score = judgment.get("overall", 0) / 5.0
    return score

@scorer  
def response_length(outputs: str) -> bool:
    """Basic check: response is non-empty and substantial."""
    if not outputs or outputs.startswith("Error:"):
        return False
    return len(outputs.strip()) >= 50

# Only include LLM judge if Anthropic is available
if ANTHROPIC_AVAILABLE:
    SCORERS = [response_quality, response_length]
    print(f"Scorers: response_quality (LLM judge), response_length")
else:
    SCORERS = [response_length]
    print(f"Scorers: response_length only (Anthropic not available)")

Scorers: response_length only (Anthropic not available)


## 4. Run Evaluation

In [7]:
def predict_fn(question: str) -> str:
    """Call the agent."""
    return agent.ask(question)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("ML Audit Agent Evaluation")

print(f"Running evaluation on {len(EVAL_DATASET)} questions...")
print("(This may take a few minutes)\n")

with mlflow.start_run(run_name="llm_judge_eval"):
    mlflow.log_param("dataset_size", len(EVAL_DATASET))
    mlflow.log_param("scorer_type", "llm_as_judge")
    
    results = mlflow.genai.evaluate(
        data=EVAL_DATASET,
        predict_fn=predict_fn,
        scorers=SCORERS,
    )

print("\nEvaluation complete!")

2026/06/16 13:04:02 INFO mlflow.tracking.fluent: Experiment with name 'ML Audit Agent Evaluation' does not exist. Creating a new experiment.


2026/06/16 13:04:02 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



2026/06/16 13:04:02 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


2026/06/16 13:04:02 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Running evaluation on 5 questions...
(This may take a few minutes)




Evaluation complete!


## 5. Results

In [8]:
print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)

if hasattr(results, 'metrics') and results.metrics:
    for k, v in results.metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.2f}")
        else:
            print(f"  {k}: {v}")
else:
    print("  (No aggregate metrics)")

print("\n" + "=" * 50)
print("DETAILED RESULTS")
print("=" * 50)

if hasattr(results, 'tables') and 'eval_results' in results.tables:
    df = results.tables['eval_results']
    # Show key columns
    cols = [c for c in df.columns if any(x in c.lower() for x in ['question', 'response', 'quality', 'length', 'score'])]
    if cols:
        display(df[cols])
    else:
        display(df)

EVALUATION RESULTS
  response_length/mean: 1.00

DETAILED RESULTS


,response_length/value,expected_response/value,response
0,True,"Agent should explain precision, recall, F1, an...",Great set of questions! Let me answer each one...
1,True,Agent should explain both concepts and their b...,Great set of questions! Let me answer all of t...
2,True,Agent should describe an investigation approac...,Great set of questions! Let me answer all of t...
3,True,"Agent should discuss model drift, threshold di...",Great set of questions! Let me answer all of t...
4,True,"Agent should provide a brief, actionable summary.",Great set of questions! Let me answer all of t...


In [9]:
# Get experiment and run info for proper link
experiment = mlflow.get_experiment_by_name("ML Audit Agent Evaluation")
if experiment:
    print(f"\nView results in MLflow UI:")
    print(f"  {MLFLOW_UI_URL}/#/experiments/{experiment.experiment_id}")
else:
    print(f"\nView results in MLflow: {MLFLOW_UI_URL}")


View results in MLflow UI:
  http://127.0.0.1:5001/#/experiments/42
